# Model initiation

In [ ]:
from extraction_tools import *
from models import *
from glob import glob

save_dir = r'D:\AIC_NEW_EXTRACTION'
keyframe_extraction_model = TransNetV2Model(device="cuda")
keyframe_feature_extraction_model = SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda")
transcription_extraction_model = FasterWhisperModel(name="turbo", device="cuda")
transcription_feature_extraction_model = BGEM3Model(name="BAAI/bge-m3", device="cuda")

# keyframe_extractor = KeyframeAndFeatureExtractor(
#     keyframe_extraction_model,
#     keyframe_feature_extraction_model,
#     save_dir
# )
transcription_extractor = TranscriptAndFeatureProcessor(
    transcription_extraction_model,
    transcription_feature_extraction_model,
    save_dir
)


[INFO] Loading quantized turbo
[INFO] turbo is on cuda device
[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

# Extract keyframes

In [ ]:
vid_paths = glob(fr"")

keyframe_extractor.extract_and_save_keyframes(
    vid_paths,
    feature_extraction_batch_size=32,
    shot_confidence_threshold=0.7,
    clustering_similarity_threshold=0.9,
    max_workers=4
)

# Extract transcripts

In [2]:
vid_paths = glob("E:/AIC_VID_ONLY/vids/*/L0*.mp4") + glob("E:/AIC_VID_ONLY/vids/*/L1*.mp4")
map_paths = glob("D:/AIC_NEW_EXTRACTION/*/L0*.csv") + glob("D:/AIC_NEW_EXTRACTION/*/L1*.csv")

if len(vid_paths) != len(map_paths): raise RuntimeError(f"Mismatch videos and maps count: {len(vid_paths)} to {len(map_paths)}")

transcription_extractor.extract_and_save_transcript(
    vid_paths,
    map_paths,
    transcription_batch_size=16,
    transcription_language="vi",
    alignment_max_time_distance=10,
    feature_extraction_batch_size=32,
    max_workers=4
)

Extracting transcripts from video:   0%|          | 0/569 [00:00<?, ?it/s]


pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 198.92it/s]









pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 674.04it/s]





















Inference Embeddings: 100%|██████████| 5/5 [00:01<00:00,  3.64it/s]



Inference Embeddings: 100%|██████████| 6/6 [00:01<00:00,  4.17it/s]



pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 699.44it/s]

pre tokenize: 100%|██████████| 5/5 [00:00<00:00, 661.25it/s]



Inference Embeddings: 100%|██████████| 6/6 [00:00<00:00,  8.02it/s]

pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 811.41it/s]2:30<4:13:46, 27.00s/it] 

pre tokenize: 100%|██████████| 5/5 [00:00<00:00, 705.85it/s]



Inference Embeddings: 100%|██████████| 6/6 [00:01<00:00,  4.96it/s]


pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 771.15it/s]3:41<3:24:49, 21.95s/it]

pre tokenize: 100%|██████████| 6/6 [00:00<00:00, 658.88it/s]



Inference Embeddings: 100%|██████████| 6/6 [00:00<00:00,  8.09i

In [4]:
feats = [np.load(path) for path in glob(r"D:\AIC_NEW_EXTRACTION\feats\bgem3_dense\L01_V001\*.npy")]
for f in feats: print(f, len(f))

[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan nan nan] 1024
[nan nan nan ... nan

# Test

In [1]:
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from extraction_tools import BEiT3Model

img_paths = [str(path) for path in list(Path(r"D:\AIC_NEW_EXTRACTION").glob("imgs/**/*.jpg"))]

beit3_model = BEiT3Model("timm/beit3_large_patch16_224.pt", device="cuda")
beit3_model.load()

[INFO] Loading quantized timm/beit3_large_patch16_224.pt
[INFO] timm/beit3_large_patch16_224.pt is on cuda device


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] Requested torchvision backend is not available. Falling back to pil backend.


In [2]:
batch_size = 1024
feats = []
for i in tqdm(list(range(0, len(img_paths), batch_size))):
    j = i + batch_size if i + batch_size < len(img_paths) else len(img_paths)
    imgs = [Image.open(path) for path in img_paths[i:j]]
    feat = beit3_model.run(imgs, batch_size=batch_size)
    for f in feat: feats.append(f)

100%|██████████| 149/149 [1:08:22<00:00, 27.53s/it]


In [4]:
import os
import numpy as np
feat_dir = Path(r"D:/AIC_NEW_EXTRACTION/feats/beit3_dense/")
for path, feat in zip(img_paths, feats):
    path = Path(path)
    if not os.path.exists(os.path.join(feat_dir, path.parts[-2])):
        os.mkdir(os.path.join(feat_dir, path.parts[-2]))
    np.save(str(Path(feat_dir) / path.parts[-2] / path.parts[-1]), feat)

In [ ]:
from glob import glob
import os
beit3_dense_paths = glob(r"D:\AIC26\feats\beit3_dense\*\*.npy")
for path in beit3_dense_paths:
    new_path = path.replace(".jpg", "")
    os.rename(path, new_path)